# 04 — Error analysis with Grad-CAM

Runs a trained checkpoint on the test set, finds the false accepts (a fake
that fooled the model into saying "real") and false rejects (a real face the
model flagged as "fake"), and shows Grad-CAM heatmaps for the worst of each —
which pixels pushed the model's decision. Needs a checkpoint from
`python -m deepfake_kyc.train` or from
[05_next_architecture_effnet_b0.ipynb](05_next_architecture_effnet_b0.ipynb),
and the `pytorch-grad-cam` package (`pip install grad-cam`) for the heatmaps.


In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "deepfake_kyc").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Open this notebook from inside the "
        "project (e.g. from notebooks/) and select the .venv kernel."
    )


PROJECT_ROOT = find_project_root()

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Set this explicitly if you want a specific run, e.g.
# CHECKPOINT = PROJECT_ROOT / "outputs" / "03_effnet_finetune" / "best.pt"
checkpoints = sorted(OUTPUTS_DIR.glob("*/best.pt")) if OUTPUTS_DIR.is_dir() else []
CHECKPOINT = checkpoints[-1] if checkpoints else None

print(f"Found {len(checkpoints)} checkpoint(s) under {OUTPUTS_DIR}")
if CHECKPOINT is not None:
    print(f"Using: {CHECKPOINT}")
else:
    print("No checkpoint found. Train a model first (see the cell above).")


## Run the model on the test set

Uses the config saved inside the checkpoint, so it matches exactly what the
model was trained with.


In [ ]:
if CHECKPOINT is not None:
    from torch.utils.data import DataLoader

    from deepfake_kyc.data import SPLIT_DIRS, build_transform, make_dataset
    from deepfake_kyc.engine import predict
    from deepfake_kyc.metrics import compute_metrics
    from deepfake_kyc.models import load_checkpoint
    from deepfake_kyc.utils import get_device

    device = get_device()
    model, cfg = load_checkpoint(CHECKPOINT, device)

    test_root = PROJECT_ROOT / cfg["data"]["root"] / SPLIT_DIRS["test"]
    test_ds = make_dataset(test_root, transform=build_transform(cfg["data"]["image_size"], train=False))
    test_loader = DataLoader(test_ds, batch_size=cfg["data"]["batch_size"], shuffle=False, num_workers=0)
    paths = [path for path, _ in test_ds.samples]

    threshold = 0.5
    probs, labels, _ = predict(model, test_loader, device)
    metrics = compute_metrics(labels, probs, threshold)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})
else:
    print("Skipped: no checkpoint available.")


## False accepts and false rejects

Positive class = fake (label 1), so this is the opposite pairing from the
usual FP/FN names: a **false accept** is a fake the model called real
(label 1, predicted 0 — feeds `far`); a **false reject** is a real face the
model called fake (label 0, predicted 1 — feeds `frr`).


In [ ]:
if CHECKPOINT is not None:
    import numpy as np

    preds = (probs >= threshold).astype(int)
    false_accepts = np.where((labels == 1) & (preds == 0))[0]  # fake mistaken for real
    false_rejects = np.where((labels == 0) & (preds == 1))[0]  # real mistaken for fake

    print(f"False accepts (fake -> real): {len(false_accepts)}")
    print(f"False rejects (real -> fake): {len(false_rejects)}")

    # Most confidently wrong first
    worst_false_accepts = sorted(false_accepts, key=lambda i: probs[i])[:6]
    worst_false_rejects = sorted(false_rejects, key=lambda i: -probs[i])[:6]
else:
    worst_false_accepts, worst_false_rejects = [], []
    print("Skipped: no checkpoint available.")


## Grad-CAM on the worst mistakes

Heatmap over the region that pushed the model towards "fake" for each image.


In [ ]:
if CHECKPOINT is not None and (len(worst_false_accepts) or len(worst_false_rejects)):
    try:
        from deepfake_kyc.explain import Explainer
    except ImportError as exc:
        raise ImportError("Grad-CAM needs pytorch-grad-cam: pip install grad-cam") from exc

    import matplotlib.pyplot as plt
    from PIL import Image

    explainer = Explainer(model, cfg["data"]["image_size"], device)
    indices = [(i, "false accept (fake\u2192real)") for i in worst_false_accepts] + \
              [(i, "false reject (real\u2192fake)") for i in worst_false_rejects]

    fig, axes = plt.subplots(len(indices), 2, figsize=(5, 2.6 * len(indices)), squeeze=False)
    for row, (idx, kind) in zip(axes, indices):
        img = Image.open(paths[idx])
        prob, overlay = explainer(img)
        row[0].imshow(img.convert("RGB"))
        row[0].set_title(kind, fontsize=9)
        row[0].axis("off")
        row[1].imshow(overlay)
        row[1].set_title(f"P(fake)={prob:.2f}", fontsize=9)
        row[1].axis("off")
    fig.tight_layout()
else:
    print("Skipped: no mistakes to show (or no checkpoint).")


## Next

Compare this run's mistakes against other experiments in
[03_results_charts.ipynb](03_results_charts.ipynb), or go back to the
experiment roadmap in
[05_next_architecture_effnet_b0.ipynb](05_next_architecture_effnet_b0.ipynb).
